# ARMxtend in 5 minutes

Synthetic baskets with planted patterns, rules with a statistical filter, the k best rules with no threshold, rare pairs, and a rule-based classifier. Everything runs on one machine; no Spark needed.

In [1]:
from ARMxtend.datasets import make_transactions, make_labeled_items, load_dataset
from ARMxtend.estimators import RuleMiner, TransactionEncoder
from ARMxtend.ARM import RuleClassifier

baskets, patterns = make_transactions(2000, 60, avg_length=7, n_patterns=8, pattern_length=3, random_state=7, return_patterns=True)
patterns[:3]

[['i09', 'i20', 'i36'], ['i00', 'i01', 'i32'], ['i04', 'i22', 'i25']]

## Rules above a support threshold, keeping only the significant ones

In [2]:
miner = RuleMiner(min_supp=0.1, min_threshold=0.8, min_lift=1.5, alpha=0.01).fit(TransactionEncoder().fit_transform(baskets))
miner.rules_[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head()

,antecedents,consequents,support,confidence,lift
0,frozenset({i22}),frozenset({i25}),0.3790,0.823018,1.830964
1,frozenset({i25}),frozenset({i22}),0.3790,0.843159,1.830964
2,"frozenset({i04, i22})",frozenset({i25}),0.3490,0.846061,1.882226
3,"frozenset({i25, i04})",frozenset({i22}),0.3490,0.853301,1.852987
4,"frozenset({i36, i09})",frozenset({i20}),0.1445,0.809524,3.782822


## The k best rules, with no support threshold to guess

In [3]:
RuleMiner(method='top_k', k=5, min_threshold=0.7).fit(baskets).rules_[['antecedents', 'consequents', 'support', 'confidence']]

,antecedents,consequents,support,confidence
0,frozenset({i00}),frozenset({i01}),0.6120,0.839506
1,frozenset({i01}),frozenset({i00}),0.6120,0.795839
2,frozenset({i09}),frozenset({i01}),0.5025,0.845960
3,frozenset({i04}),frozenset({i01}),0.4900,0.736842
4,frozenset({i09}),frozenset({i00}),0.4785,0.805556


## Rare combinations of frequent items

In [4]:
pairs = [['a', 'd']] * 15 + [['a']] * 15 + [['d']] * 15 + [['b']] * 55
RuleMiner(method='rare', max_supp=0.2, min_threshold=0.5).fit(pairs).rules_[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

,antecedents,consequents,support,confidence,lift
0,frozenset({a}),frozenset({d}),0.15,0.5,1.666667
1,frozenset({d}),frozenset({a}),0.15,0.5,1.666667


## Rules as a classifier (CBA)

In [5]:
weather = load_dataset('weather')
tennis = RuleClassifier(min_supp=0.1, min_conf=0.8).fit(weather.data, weather.target)
tennis.rules_[['antecedents', 'class', 'confidence']]

,antecedents,class,confidence
0,frozenset({outlook=overcast}),yes,1.0
1,"frozenset({humidity=normal, windy=false})",yes,1.0
2,"frozenset({outlook=sunny, humidity=high})",no,1.0
3,"frozenset({outlook=rainy, windy=false})",yes,1.0
4,"frozenset({outlook=rainy, windy=true})",no,1.0


In [6]:
items, labels = make_labeled_items(1500, noise=0.05, random_state=3)
RuleClassifier(min_supp=0.05, min_conf=0.7).fit(items[:1000], labels[:1000]).score(items[1000:], labels[1000:])

0.982